In [1]:
from utils.generic_utils import load_all_games_csv, predict_lr
from utils.elo_tracker_utils import evaluate_elo_prob_func
import numpy as np

# Model vs 538

This will compare my best model to the 538 baseline, for all seasons and the 2024 season.

## Get Games

In [ ]:
games = load_all_games_csv('../data/gameinfo_cleaned.csv', preprocess=True)
#games = games[games['season']==2025]
games.head()

/Users/lancehendricks/Documents/College Coding/ML/Elo Ratings/analysis/src/utils/generic_utils.py:30: DtypeWarning: Columns (10,11,13,17,19,20,21,27,28) have mixed types. Specify dtype option on import or set low_memory=False.
  all_games = pd.read_csv(filename)


,visteam,hometeam,site,date,number,starttime,daynight,innings,tiebreaker,usedh,...,homerestdays,visrestdays,homepitcherrgs,vispitcherrgs,hometeamrgs,visteamrgs,homepitcherminusteamrgs,vispitcherminusteamrgs,homelastkwinpct,vislastkwinpct
gid,,,,,,,,,,,,,,,,,,,,,
CIN189804150,CL4,CIN,CIN05,18980415,0.0,0:00PM,day,NaN,NaN,False,...,NaN,NaN,39.0,39.0,39.0,39.0,0.0,0.0,NaN,NaN
LS3189804150,PIT,LS3,LOU03,18980415,0.0,0:00PM,day,NaN,NaN,False,...,NaN,NaN,39.0,39.0,39.0,39.0,0.0,0.0,NaN,NaN
SLN189804150,CHN,SLN,STL05,18980415,0.0,0:00PM,day,NaN,NaN,False,...,NaN,NaN,39.0,39.0,39.0,39.0,0.0,0.0,NaN,NaN
BLN189804160,WSN,BLN,BAL07,18980416,0.0,0:00PM,day,NaN,NaN,False,...,NaN,NaN,39.0,39.0,39.0,39.0,0.0,0.0,NaN,NaN
CIN189804160,CL4,CIN,CIN05,18980416,0.0,0:00PM,day,NaN,NaN,False,...,1.0,1.0,39.0,39.0,65.9,64.9,-26.9,-25.9,1.0,0.0


In [3]:
# Max rest days
games['visrestdays'] = games['visrestdays'].apply(lambda x: min(3,x))
games['homerestdays'] = games['homerestdays'].apply(lambda x: min(3,x))


In [4]:
# Take cube root of distance traveled
games['homedistancetraveled'] = games['homedistancetraveled']**(1/3)
games['visdistancetraveled'] = games['visdistancetraveled']**(1/3)

In [5]:
# Take square root of margin of victory
games['marginofvictory'] = np.sqrt(games['marginofvictory'])
games.head()

,visteam,hometeam,site,date,number,starttime,daynight,innings,tiebreaker,usedh,...,homerestdays,visrestdays,homepitcherrgs,vispitcherrgs,hometeamrgs,visteamrgs,homepitcherminusteamrgs,vispitcherminusteamrgs,homelastkwinpct,vislastkwinpct
gid,,,,,,,,,,,,,,,,,,,,,
CIN189804150,CL4,CIN,CIN05,18980415,0.0,0:00PM,day,NaN,NaN,False,...,3.0,3.0,39.0,39.0,39.0,39.0,0.0,0.0,NaN,NaN
LS3189804150,PIT,LS3,LOU03,18980415,0.0,0:00PM,day,NaN,NaN,False,...,3.0,3.0,39.0,39.0,39.0,39.0,0.0,0.0,NaN,NaN
SLN189804150,CHN,SLN,STL05,18980415,0.0,0:00PM,day,NaN,NaN,False,...,3.0,3.0,39.0,39.0,39.0,39.0,0.0,0.0,NaN,NaN
BLN189804160,WSN,BLN,BAL07,18980416,0.0,0:00PM,day,NaN,NaN,False,...,3.0,3.0,39.0,39.0,39.0,39.0,0.0,0.0,NaN,NaN
CIN189804160,CL4,CIN,CIN05,18980416,0.0,0:00PM,day,NaN,NaN,False,...,1.0,1.0,39.0,39.0,65.9,64.9,-26.9,-25.9,1.0,0.0


## Evaluate my Model

In [6]:
w = np.array([[ 1.        ],
       [27.1440666 ],
       [ 4.81476328],
       [-0.30523283],
       [ 1.58004319]])

In [7]:
bce, accuracy = evaluate_elo_prob_func(games, lambda home_elo, away_elo, game: predict_lr(home_elo, away_elo, game, w), K=3, use_margin_of_victory=True, skip_first_n=0)
print(f"BCE: {bce}")
print(f"Accuracy: {accuracy}")

BCE: 0.6753945081099757
Accuracy: 0.5745099839338995


In [8]:
bce, accuracy = evaluate_elo_prob_func(games, lambda home_elo, away_elo, game: predict_lr(home_elo, away_elo, game, w), K=3, use_margin_of_victory=True, skip_first_n=0, years=set(range(2021,2026)))
print(f"BCE: {bce}")
print(f"Accuracy: {accuracy}")

BCE: 0.6772416003536925
Accuracy: 0.5709805216242985


In [9]:
bce, accuracy = evaluate_elo_prob_func(games, lambda home_elo, away_elo, game: predict_lr(home_elo, away_elo, game, w), K=3, use_margin_of_victory=True, skip_first_n=0, years={2025})
print(f"BCE: {bce}")
print(f"Accuracy: {accuracy}")

BCE: 0.6770029763226537
Accuracy: 0.5623603039785426


## Evaluate 538 Model

In [10]:
w_538 = np.array([[ 1.        ],
       [24 ],
       [ 2.3],
       [-0.31],
       [ 4.7]])

In [11]:
bce, accuracy = evaluate_elo_prob_func(games, lambda home_elo, away_elo, game: predict_lr(home_elo, away_elo, game, w_538), K=3, use_margin_of_victory=False, skip_first_n=0)
print(f"538 BCE: {bce}")
print(f"538 Accuracy: {accuracy}")

538 BCE: 0.6810046511712547
538 Accuracy: 0.5670002295157218


In [12]:
bce, accuracy = evaluate_elo_prob_func(games, lambda home_elo, away_elo, game: predict_lr(home_elo, away_elo, game, w_538), K=3, use_margin_of_victory=False, skip_first_n=0, years=set(range(2021,2026)))
print(f"538 BCE: {bce}")
print(f"538 Accuracy: {accuracy}")

538 BCE: 0.6833507126696382
538 Accuracy: 0.5586827335754374


In [13]:
bce, accuracy = evaluate_elo_prob_func(games, lambda home_elo, away_elo, game: predict_lr(home_elo, away_elo, game, w_538), K=3, use_margin_of_victory=False, skip_first_n=0, years={2025})
print(f"538 BCE: {bce}")
print(f"538 Accuracy: {accuracy}")

538 BCE: 0.6838535904396755
538 Accuracy: 0.5556548949485919
